# <font color='green'>PageRank</font>

### We'll start off by importing what we need

In [ ]:
import numpy as np
import numpy.linalg as npla

import scipy.sparse as sparse
from scipy.sparse.linalg import eigs
import scipy

import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import axes3d
%matplotlib inline

np.set_printoptions(precision = 3)

# <font color='blue'>PageRank</font>
### Based on the example given in Lecture 15 (PageRank)

**<font color='red'>Recall from the lecture</font>** that you are going to need to take the given **adjacency matrix $E$**, 

then develop the **link matrix $LM$** = $E$/**outdegree**, 

then finally develop the **Markov Matrix $M$** = $p.LM + \delta$.

## Let's define a function `make_M_from_E`:
### This takes an adjacency matrix E and creates a Markov Matrix M

In [ ]:
def make_M_from_E(E, m = 0.15):
    """
    Make the PageRank matrix from the adjacency matrix (E) of a graph.
    m is the probability that and arbitrary page is chosen (i.e. the random surfer model)
    m has a default value of 0.15
    ***E must NOT be a sparse matrix***
    """
    # Calculate the outdegree of each node:
    #   Do this by adding the numbers in each column:
    outdegree = np.sum(E, axis=0)  
    
    # Check for hanging nodes, i.e. for where outdegree is zero.
    n = E.shape[0]
    for j in range(n):
        if outdegree[j] == 0:
            E[:, j] = np.ones(n)
            E[j, j] = 0
    
    # Calculate LM = E / outdegree:
    LM = E / np.sum(E, axis=0)
    
    # Calculate M = p. LM + delta
    delta = m * np.ones((n,n)) / n
    M = (1 - m) * LM + delta
    return M

### Let's load simple networks found in files `PageRankEG1.npy` and `PageRankEG2.npy`

## `PageRankEG1.npy`

In [ ]:
# Load file

E = np.load('PageRankEG1.npy')
print(E)

# ALTERNATIVE RUN: 
#   Try this with a dangling node, to see how the make_M_from_E function deals with it:
#   e.g.   E = np.array([[0,0,0,1],[1,0,0,0],[1,1,0,1],[1,1,0,0]])

In [ ]:
#spy plot example using E (simple)

%matplotlib inline
plt.spy(E)

**We'll calculate the Markov Matrix M, per the explanation given in lecture...**
1. Find the Link Matrix (LM)
2. Calculate M from that

### Our `make_M_from_E()` function will do this for us!

In [ ]:
M = make_M_from_E( E )
print(M)

### <font color='blue'>Now for the calculations we need for PageRank...</font>

1. Get the eigenvalues, find the position of the largest one (it'll be where lambda = 1)
2. Get the eigenvector at that position
3. Sort the values in that vector in reverse order
4. <font color='red'>THAT'S the PageRank order of the nodes!</font>
5. Profit??

**1.** Get the eigenvalues, find the position of the largest one (it'll be where lambda = 1)

*if you're calling the npla.eig() function, you might as well calculate the eigenvectors*

In [ ]:
# 1. Get the eigenvalues of M:

d,V = npla.eig(M)
# Note that eigenvalues CAN BE complex numbers!!!
# But we only want to deal with their real parts...

print('Eigenvalues       :', d)
print('Eigenvalues (real):', d.real)

In [ ]:
# Find the LARGEST (real portion) eigenvalue
# Because of the nature of the Markov Matrix,
# this will always be where lambda = 1.

max_d = max(d.real)  # We expect this to be 1.
print('Max Eigenvalue of M (should be 1):', max_d) 

In [ ]:
# Now let's see WHERE this eigenvalue is - by using np.where()
# Note that np.where() returns a TUPLE, not an integer result

print("This is what np.where(d == max(d)) returns:", np.where(d == max(d)))

# So the position is in the array() portion of the result, so to extract it, we can do this:
pos_max_d = int(np.where(d == max(d))[0])

print('\nThe max. eigenvalue (1) is located in position ', pos_max_d)

print('\nAll eigenvectors of M (real values only):\n', V.real, '\n')

In [ ]:
# Introducing argsort() function - look at this example:

v = [33, 11, 22]
print("v =", v)
print("If you wanted vector v sorted, then the order to use is:", np.argsort(v))

**2.** Get the eigenvector at that position.

In [ ]:
v = V[:, pos_max_d].real
print('Eigenvector at position', pos_max_d, 'is::::::::::::::', v)

**3.** Sort the values in that vector in reverse order.

In [ ]:
# In Python, [::-1] indexing means reverse numbers in container
ranked = np.argsort(v)[::-1]

print('Eigenvector at position', pos_max_d, 'sorted in order:', v[ranked], '\n')
print(ranked, 'is the same as the vertices\' (nodes\') PageRank order!!!')

## `PageRankEG2.npy`

**Let's repeat this process with another example...**

In [ ]:
E = np.load('PageRankEG2.npy')
print(E)

#spy plot example using E (simple)

%matplotlib inline
plt.spy(E)
print()

In [ ]:
M = make_M_from_E(E)
print('M =\n', M, '\n')

# DIRECT METHOD:
d,V = npla.eig(M)
print('Eigenvalues:', d)

max_d = max(d)
print('Max Eigenvalue:', max_d) # It will ALWAYS be 1.0

# Note that np.where() rexturns a TUPLE, not an integer result
pos_max_d = int(np.where(d == max(d))[0])
print('This is located in position %d\n'%pos_max_d)

# Let's look at the ranking of values the eigenvector corresponding to the largest eigenvalue:
v = V[:, pos_max_d].real
ranked = np.argsort(v)[::-1]
#print(V)
print('Eigenvector at position', pos_max_d, 'is: ', v)
print('The reverse ranking of these values is:', ranked)
print('This is the same as the vertices\' PageRank order!!!')

# <font color='red'>Ok, let's go bigger...!!!</font>
*(bigger graphs, that is...)*

## `PageRankEG3.npy`
### *This is an example with Harvard.com webpages (includes 500 nodes and their labels)*

In [ ]:
E = np.load('PageRankEG3.npy')
print(E)
print("\nThe size of this matrix is:", E.shape)

In [ ]:
# spy-plot!

%matplotlib inline
plt.spy(E)

## <font color="blue">Examine the data a little bit first...</font>
**Let's read the node/vertex labels (they're in the file we loaded)** 

and let's see a few of them (remember, there's 500 of them)

In [ ]:
# Look at the first 10 listed URLs in this dataset

with open('PageRankEG3.nodelabels') as f:
    labels = f.read().splitlines()
    
for i in range(10):
    print(i, labels[i])

In [ ]:
# Let's create the Markov Matrix

M = make_M_from_E(E)
print(M, '\n')
print(M.shape)

In [ ]:
# Calculate the eigenvalues/eigenvectors of M
d, V = npla.eig(M)

max_d = max(d.real)
print('Max Eigenvalue:', max_d) # It will ALWAYS be 1.0

# Note that np.where() returns a TUPLE, not an integer result
pos_max_d = int(np.where(d == max(d))[0])
print('This is located in position %d\n'%pos_max_d)

print('Eigenvector at that position:\n', V[:, pos_max_d].real)
# Careful, there's 500 of these!!!

In [ ]:
# Designate the eigenvector of M as the PageRank solution x
# Sort them in reverse numerical order using .argsort() function
# put the result in a Python list called perm[]

x = V[:, pos_max_d].real
perm = np.argsort(x)[::-1]

print('The reverse numerical order (i.e. PageRank order) of the first 10 nodes is:\n', perm[:10])

In [ ]:
# Print the node labels associated with the first 10 nodes in the perm[] list
# (this file HAPPENS to have labels associated with every vertex/node)

for i in range(490, 500):
    print(i, labels[perm[i]])
    
# These are the PageRanked returned results!

# <font color='blue'>Very Large Data Set Example...</font>

## Google Web small subset example (>900k pages)
### *This is where our usual approach will break down...* :(

In [ ]:
E = scipy.sparse.load_npz('webGoogle.npz')   # Note this is already a SPARSE matrix in the file...
print(E.shape)

# print(E)

In [ ]:
# Too big, won't plot in a comprehensive way

%matplotlib inline
plt.spy(E)

In [ ]:
# Note: if you want to do eigen. functions on sparse matrices,
# you should use      scipy.sparse.linalg.eigs()
# Recall, we imported scipy.sparse as sparse...

d, V = scipy.sparse.linalg.eigs(E)

# Function returns only SIX (6) e-values/vectors by default. AND it took a LONG TIME!
# It is not always possible to compute all eigenvectors of a large sparse matrix.
print("The eigenvector matrix returned is only this big:", V.shape)

# We can't be sure which e-vector to look at! Is it the 1st one again?
x = V[:, 0]
perm = np.argsort(x)[::-1]

print('Is this (eigenvector at 0) the answer? Maybe... maybe not!...\n', perm[:10])

# <font color="blue">We HAVE to use a different approach...
    
## The Iterative Power Method
### We'll call this PageRank1() method
*You'll be asked to modify this for very large data files in your homework...*

In [ ]:
# Let's go back to the easy example:

E = np.load('PageRankEG1.npy')
print("E=\n", E)

M = make_M_from_E(E)
print("\nM=\n",M)

In [ ]:
def power_page_rank( M, iters = 30 ):
    x = np.ones(M.shape[0])
    for i in range( iters ):
        x = (M @ x) / npla.norm(M @ x)
    return x

print(power_page_rank(M))


In [ ]:
# Iterative method begins here:
# Set x with an initial value

x = np.ones(M.shape[0])
print(x)

In [ ]:
# Power Method
x = np.ones(M.shape[0])
NofIterations = 25
for i in range(NofIterations):
    xp = x
    x = M @ x
    x = x / npla.norm(x)
    relresnorm = npla.norm(x - xp)/npla.norm(xp)
    print(i, x, relresnorm)

In [ ]:
# Let's test to see if x is indeed a normalized vector:

print(npla.norm(x) == 1.)

In [ ]:
# Compare with numpy eigenvalue function:
# (now this is 'cheating' b/c we already did this exercise and we know to look at e-vector_0)

d, V = npla.eig(M)

# print the 0th column of the eigenvectors
xe = V[:, 0].real
print(xe)

In [ ]:
# How different is this from our iterative x?

print(npla.norm(xe - x)/npla.norm(x))

# Function pagerank1()
Use this in your next homework

This function takes in $E$, an adjacency matrix, as a REGULAR matrix `np.array()` datatype and calculates the PageRank order of all the vertices/nodes

In [ ]:
def pagerank1(E, return_vector = False, max_iters = 1000, tolerance = 1e-8):
    """compute page rank from dense adjacency matrix

    Inputs:
      E: adjacency matrix with links going from cols to rows.
         E is a matrix of 0s and 1s, where E[i,j] = 1 means 
         that web page (vertex) j has a link to web page i.
      return_vector = False: If True, return the eigenvector as well as the ranking.
      max_iters = 1000: Maximum number of power iterations to do.
      tolerance = 1e-6: Stop when the eigenvector norm changes by less than this.
      
    Outputs:
      ranking: Permutation giving the ranking, most important first
      vector (only if return_vector is True): Dominant eigenvector of PageRank matrix

    This computes page rank by the following steps:
    1. Add links from any dangling vertices to all vertices.
    2. Scale the columns to sum to 1.
    3. Add a constant matrix to represent jumping at random 15% of the time.
    4. Find the dominant eigenvector with the power method.
    5. Sort the eigenvector to get the rankings.

    The homework problem asks you to rewrite this code so
    it takes input E as a scipy csr_sparse matrix, and then never creates 
    a full matrix or any large matrix other than E.
    """
    
    if type(E) is not np.ndarray:
        print('Warning, converting input from type', type(E), 'to dense array.')
        E = E.toarray()
                
    nnz = np.count_nonzero(E)       # This call for sparse E may be different
    outdegree = np.sum(E, axis=0)   # This call for sparse E may be different
    nrows, n = E.shape

    assert nrows == n, 'E must be square'
    assert np.max(E) == 1 and np.sum(E) == nnz, 'E must contain only zeros and ones'
    
    #  1. Add links from any dangling vertices to all other vertices.
    #     E + F will be the matrix with the added links.

    F = np.zeros((n,n))
    for j in range(n):
        if outdegree[j] == 0:
            F[: , j] = np.ones(n)
            F[j , j] = 0
    
    #  2. Scale the columns to sum to 1 (i.e. normalization of E+F) - Link Matrix:

    A = (E + F) / np.sum(E + F, axis=0)
    
    #  3. Add a constant matrix to represent jumping at random 15% of the time.

    S = np.ones((n,n)) / n
    m = 0.15
    M = (1 - m) * A + m * S
    
    #  4. Find the dominant eigenvector using the Power Method.
    #  Start with a vector all of whose entries are equal.

    e = np.ones(n)
    v = e / npla.norm(e)  # a better first-guess than an "all 1s" vector

    for iteration in range(max_iters):
        oldv = v
        
        v = M @ v
        eigval = npla.norm(v)
        v = v / eigval
        
        if npla.norm(v - oldv) < tolerance:
            break
    
    if npla.norm(v - oldv) < tolerance:
        print('Dominant eigenvalue is %f after %d iterations.\n' % (eigval, iteration+1))
    else:
        print('Did not converge to tolerance %e after %d iterations.\n' % (tolerance, max_iters))

    # Check that the eigenvector elements are all the same sign, and make them positive
    assert np.all(v > 0) or np.all(v < 0), 'Error: eigenvector is not all > 0 or < 0'
    vector = np.abs(v)
        
    #  5. Sort the eigenvector and reverse the permutation to get the rankings.
    ranking = np.argsort(vector)[::-1]

    if return_vector:
        return ranking, vector
    else:
        return ranking

# end of pagerank1()

### Running the Iterative Approach on the previous data sets EG1, EG2, EG3

**EG1**

In [ ]:
# Non-iterative (direct) approach gave us:  
# r = [0 2 3 1]

E = np.load('PageRankEG1.npy')
r, v = pagerank1(E, return_vector = True)
print('r =', r)
print('v =', v)

**EG2**

In [ ]:
# Non-iterative approach gave us:  
# r = [3 2 0 1 4]

E = np.load('PageRankEG2.npy')
r, v = pagerank1(E, return_vector = True)
print('r =', r)
print('v =', v)

**EG3** (aka the Harvard data set)

In [ ]:
# Recall: 500x500 matrix

# Non-iterative approach gave us:  
# r = [  0   9  41 129  17  14   8  16  45  12]
# For the 1st 10 nodes

E = np.load('PageRankEG3.npy')
r, v = pagerank1(E, return_vector = True)
print('r =', r[:10])
print('v =', v[:10])

**EG4** (aka the Google data set with the sparse matrix)

(it won't work because $E$ is not an `np.array()` type!!)

In [ ]:
# Recall: size is almost 1 million x 1 million
# It won't work with pagerank1()
# Because forcing it to work with a dense matrix 
# (recall that the function ensures the input matrix is dense, by conversion if needed)
# will quickly eat up your computer's data memory resources

# Note E is already a SPARSE matrix by definition...
E = scipy.sparse.load_npz('webGoogle.npz')   

r, v = pagerank1(E, return_vector = True)
print('r =', r[:10])
print('v =', v[:10])

## What follows is `pagerank2()`:
## an incomplete version of `pagerank1()` that can deal with sparse matrices

### <font color='red'>This is your homework this week...!</font>

In [ ]:
def pagerank2(E, return_vector = False, max_iters = 1000, tolerance = 1e-6, m = 0.15):
    """compute page rank from dense adjacency matrix
    Inputs:
      E: adjacency matrix with links going from cols to rows.
         E is a matrix of 0s and 1s, where E[i,j] = 1 means 
         that web page (vertex) j has a link to web page i.
      return_vector = False: If True, return the eigenvector as well as the ranking.
      max_iters = 1000: Maximum number of power iterations to do.
      tolerance = 1e-6: Stop when the eigenvector norm changes by less than this.
      m = 0.15: default
      
    Outputs:
      ranking: Permutation giving the ranking, most important first
      vector (only if return_vector is True): Dominant eigenvector of PageRank matrix
    This computes page rank by the following steps:
    1. Add links from any dangling vertices to all vertices.
    2. Scale the columns to sum to 1.
    3. Add a constant matrix to represent jumping at random 15% of the time.
    4. Find the dominant eigenvector with the power method.
    5. Sort the eigenvector to get the rankings.
    
    The function takes input E as a scipy csr_sparse matrix, and then never creates 
    a full matrix or any large matrix other than E.
    """

# HERE ARE ALL THE TASKS YOU NEED TO FIGURE OUT!
#################################################

# 1. Check if E is a square matrix and that its entries are ONLY 1s and 0s, otherwise end this

# 2. Check if E is a sparse matrix. If NOT, then convert it into one.
 
# 3. Calculate the outdegree 

# 4. Get set up for the power iteration:
#       create an initial vector (all 1s)
#       make sure you know where outdegree is 0

    for iteration in range(max_iters):
        oldv = v

      #Remember: The equation we are trying to solve is: MV = (1−m)(EV+FV) + m*SV,
      #     where:  SV is the average of vector v
      #             EV is matrix E multiplied by the normalized version of vector v
      #             FV is a matrix that accounts for dangling vertices (i.e. where outdegree is 0 in E). 
      #                 Note that if there are no dangling nodes in E, F will always be 0 (easy case).
      
      
      #Part 1: SV -- This last (and easiest) part of the equation essentially is just an 
      # "average" operator on the matrix v. By this I mean that every item in the vector is just 
      # The sum of the vector components divided by the number of nodes, which is an average.
      # You can find SV with a simple and efficient alternative to actually making the dense S matrix and  
      # multiplying it by v to get the same result.
      

      #Part 2: EV  -- Multiply E by normalized v vector
      
      
      #Part 3: FV -- In order to avoid having to actually construct the (likely sparse) F matrix,
      # We will take advantage of the unique properties. Essentially we know that for a dangling node,
      # the probability of landing on any other node is evenly split.  This translates to the values
      # at the index of the dangling nodes in FV being one normalized unit of v less than all the other
      # values at non-dangling indicies. You can do this in as little as 3 lines of code and thus 
      # calculate the FV vector without any matrix multiplication. 
      
      #Part 4: Calculate MV per the formula
    
    
    #5. And now you should be back to exactly where we were with pagerank1(), 
    #   so finish this up based on what we did in that function.
    
      v = MV
      eigval = npla.norm(v)
      v = v / eigval
      
    # finish it up here, just like in pagerank1()